# DINO Fine-tuning for Product Recognition

In this notebook we implemented a **DINO (Distillation with NO Labels)** fine-tuning approach combined with YOLO object detection.

### Components of Pipeline :
1. **DINO Fine-tuning**: Train student-teacher model for feature extraction
2. **YOLO Detection**: Detect products in images
3. **Embedding Extraction**: Extract DINO embeddings for detected crops
4. **FAISS Indexing**: Build efficient similarity search index

### Dataset Used : "https://www.kaggle.com/datasets/diyer22/retail-product-checkout-dataset"

## Part 1: DINO Model Training

### 1.1 Setup and Dependencies
Import required libraries for model training, data handling, and neural network operations.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import json
import cv2
import numpy as np
import faiss
import torchvision.transforms as T
from tqdm import tqdm
from ultralytics import YOLO
from collections import defaultdict, Counter



### 1.2 Data Augmentation Transforms
Define DINO-specific augmentation strategy:
- **Global crops**: 2 large crops at 224x224 for multi-view learning
- **Local crops**: 4 smaller crops (98x98) for local context
- **Augmentations**: Resize, horizontal flip, color jitter, Gaussian blur, normalization

This multi-scale approach helps the model learn both global and local features.

In [27]:
class DINOTransform:
    def __init__(self):
        self.global_transform = T.Compose([
            T.RandomResizedCrop(224, scale=(0.5, 1.0)),
            T.RandomHorizontalFlip(),
            T.ColorJitter(0.2,0.2,0.2,0.05),
            T.GaussianBlur(3),
            T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])

        self.local_transform = T.Compose([
            T.RandomResizedCrop(98, scale=(0.2, 0.5)),
            T.RandomHorizontalFlip(),
            T.ColorJitter(0.2,0.2,0.2,0.05),
            T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])

        self.local_crops_number = 4

    def __call__(self, image):
        crops = []
        crops.append(self.global_transform(image))
        crops.append(self.global_transform(image))

        for _ in range(self.local_crops_number):
            crops.append(self.local_transform(image))

        return crops

### 1.3 Custom Dataset for RPC Data
Dataset class for loading retail product images:
- Loads JPEG images from root directory
- Extracts product label from filename format: `image_{label}.jpg`
- Applies DINOTransform to generate multiple crops per image

In [28]:
class RPCDataset(Dataset):
    def __init__(self, root):
        self.root = root
        self.files = [f for f in os.listdir(root) if f.endswith(".jpg")]
        self.transform = DINOTransform()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]
        path = os.path.join(self.root, file)

        img = Image.open(path).convert("RGB")

        label = int(file.split("_")[1].split(".")[0])

        crops = self.transform(img)

        return crops, label

### 1.4 Custom Collate Function
Processes batch of multi-crop samples:
- Groups all crops by crop type (global, local)
- Stacks crops into tensors for efficient GPU processing
- Returns list of cropped tensors and corresponding labels

In [29]:
def dino_collate(batch):
    all_crops = list(zip(*[item[0] for item in batch]))
    all_crops = [torch.stack(crops) for crops in all_crops]

    labels = torch.tensor([item[1] for item in batch])

    return all_crops, labels

### 1.5 Data Loading
Initialize DataLoader:
- **Batch size**: 16 samples
- **Shuffle**: True for random sampling during training
- **Drop last**: True to avoid incomplete batches
- **Num workers**: 2 for parallel data loading

**Note**: Please update `DATA_PATH` to correct Dataset location

In [ ]:
DATA_PATH = "/kaggle/input/datasets/sarthakd25/cropped-rpc/kaggle/working/cropped_dataset" 

dataset = RPCDataset(DATA_PATH)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    drop_last=True,
    collate_fn=dino_collate
)

### 1.6 Student-Teacher Model Architecture
DINO uses a student-teacher model where:
- **Student model**: ViT-S/14 updated via backprop
- **Teacher model**: ViT-S/14 updated via momentum (exponential moving average)

This asymmetric design prevents model collapse while enabling effective self-supervised learning.

In [31]:
device = "cuda" if torch.cuda.is_available() else "cpu"

student = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
teacher = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)

### 1.7 Model Fine-tuning Setup
Configure trainable parameters:
- **Student**: Unfreeze last 2 transformer blocks (10-11) for fine-tuning
  - This balances learning capacity with computational efficiency
- **Teacher**: Freeze all parameters (updated via momentum only)

In [32]:
for name, param in student.named_parameters():
    if "blocks.10" in name or "blocks.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

for p in teacher.parameters():
    p.requires_grad = False

### 1.8 Projection Heads
Add projection heads for both student and teacher:
- **Input dimension**: 384 (ViT-S/14 embedding)
- **Output dimension**: 256 (projection dimension)
- **Architecture**: Linear → GELU → Linear
- Teacher head is frozen (no gradients)

In [33]:
class DINOHead(nn.Module):
    def __init__(self, in_dim=384, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.GELU(),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        return self.net(x)

student_head = DINOHead().to(device)
teacher_head = DINOHead().to(device)

for p in teacher_head.parameters():
    p.requires_grad = False

### 1.9 Loss Functions

#### DINO Contrastive Loss
Knowledge distillation loss comparing student and teacher:
- **Teacher temperature** (`t_temp=0.04`): Sharpens teacher probabilities
- **Student temperature** (`s_temp=0.1`): Controls student softness
- Minimizes KL divergence between student and teacher predictions

In [34]:
class DINOLoss(nn.Module):
    def __init__(self, t_temp=0.04, s_temp=0.1):
        super().__init__()
        self.t_temp = t_temp
        self.s_temp = s_temp

    def forward(self, student_out, teacher_out):
        loss = 0
        n = 0

        for t in teacher_out:
            t = F.softmax(t.detach() / self.t_temp, dim=-1)

            for s in student_out:
                s = F.log_softmax(s / self.s_temp, dim=-1)
                loss += torch.mean(torch.sum(-t * s, dim=-1))
                n += 1

        return loss / n

#### Product Similarity Loss
Custom loss to enforce metric learning:
- **Same product pairs**: Minimize distance (attract)
- **Different product pairs**: Push away from margin 0.3 (repel)
- Helps model learn discriminative embeddings for product classification

In [35]:
def product_loss(embeddings, labels):
    embeddings = F.normalize(embeddings, dim=1)

    loss = 0
    count = 0

    for i in range(len(embeddings)):
        for j in range(len(embeddings)):
            if i == j:
                continue

            sim = F.cosine_similarity(
                embeddings[i].unsqueeze(0),
                embeddings[j].unsqueeze(0)
            )

            if labels[i] == labels[j]:
                loss += (1 - sim)
            else:
                loss += F.relu(sim - 0.3)

            count += 1

    return loss / count

### 1.10 Optimizer Configuration
Setup optimizer for student model parameters:
- **Type**: AdamW (Adam with weight decay)
- **Learning rate**: 1e-4 (low rate for fine-tuning)
- **Parameters**: Student backbone + projection head

In [36]:
optimizer = torch.optim.AdamW(
    list(student.parameters()) + list(student_head.parameters()),
    lr=1e-4
)

### 1.11 Teacher Model Update
Momentum-based teacher update:
- **Momentum coefficient** (m=0.996): High momentum means slow updates
- Formula: `teacher_weight = m * teacher_weight + (1-m) * student_weight`
- Keeps teacher stable while incorporating student improvements gradually

In [37]:
def update_teacher(student, teacher, student_head, teacher_head, m=0.996):
    for ps, pt in zip(student.parameters(), teacher.parameters()):
        pt.data = m * pt.data + (1 - m) * ps.data

    for ps, pt in zip(student_head.parameters(), teacher_head.parameters()):
        pt.data = m * pt.data + (1 - m) * ps.data

### 1.12 Training Loop
Main training procedure:

**Steps per iteration:**
1. Forward pass through all crops (student)
2. Forward pass through global crops only (teacher)
3. Compute DINO distillation loss
4. Compute product similarity loss
5. Combine losses: `loss = loss_dino + λ_p × loss_prod` (λ_p=0.5)
6. Backward pass with gradient clipping (max norm=1.0)
7. Optimizer step and teacher update

**Hyperparameters:**
- Epochs: 10
- Lambda (product loss weight): 0.5

In [38]:
criterion = DINOLoss()
lambda_p = 0.5

epochs = 10

for epoch in range(epochs):
    student.train()
    student_head.train()

    for crops, labels in loader:
        crops = [c.to(device) for c in crops]
        labels = labels.to(device)

        global_crops = crops[:2]

        # student forward
        student_out = [student_head(student(c)) for c in crops]

        # teacher forward
        with torch.no_grad():
            teacher_out = [teacher_head(teacher(c)) for c in global_crops]

        # losses
        loss_dino = criterion(student_out, teacher_out)

        emb = student_out[0]
        loss_prod = product_loss(emb, labels)

        loss = loss_dino + lambda_p * loss_prod

        optimizer.zero_grad()
        loss.backward()

        # stability
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)

        optimizer.step()

        update_teacher(student, teacher, student_head, teacher_head)

    print(f"Epoch {epoch}, Total: {loss.item():.4f}, DINO: {loss_dino.item():.4f}, Prod: {loss_prod.item():.4f}")

### 1.13 Save Trained Model
Save all model components for inference:

In [39]:
torch.save({
    'student': student.state_dict(),
    'student_head': student_head.state_dict(),
    'teacher': teacher.state_dict(),
    'teacher_head': teacher_head.state_dict()
}, "/kaggle/working/dino_rpc_model.pth")

In [40]:
!pip install ultralytics
!pip install faiss-cpu

---

## Part 2: Combined YOLO + DINO Inference Pipeline

This section implements the complete inference pipeline combining:
- **YOLO**: Object detection to find products
- **DINO**: Feature extraction for detected products
- **FAISS**: Efficient similarity search for product identification

### 2.1 Environment and Dependencies

In [ ]:

TRAIN_IMG_DIR = "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/val2019"
TRAIN_JSON = "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/instances_val2019.json"
TEST_IMG_DIR = "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/test2019"
TEST_JSON = "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/instances_test2019.json"

YOLO_CKPT = "/kaggle/input/models/khushalnikam/yolocheckp/pytorch/default/1/best.pt"
DINO_CKPT = "/kaggle/input/models/khushalnikam/mycheckpoin/pytorch/default/1/dino_rpc_model.pth"

# Output files
FAISS_INDEX_PATH = "product_embeddings.index"
LABEL_MAP_PATH = "embedding_labels.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

### 2.2 Model Loading and Helper Functions
Load YOLO detector and DINO feature extractor with utility functions:
- **YOLO**: Pre-trained object detector for product localization
- **DINO**: Fine-tuned ViT-S/14 model for feature extraction
- **get_embedding()**: Converts BGR crop to normalized DINO embedding
- **calculate_iou()**: Computes IoU between detection and ground truth boxes

In [ ]:
print("Loading Models...")
yolo_model = YOLO(YOLO_CKPT)

# Load DINOv2 Student model
student = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(DEVICE)
checkpoint = torch.load(DINO_CKPT, map_location=DEVICE)
student.load_state_dict(checkpoint['student'])
student.eval()

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def get_embedding(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    input_tensor = transform(pil_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        emb = student(input_tensor).cpu().numpy()[0]
    return (emb / np.linalg.norm(emb)).astype('float32')

def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)
    boxAArea = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
    boxBArea = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)
    return interArea / float(boxAArea + boxBArea - interArea)


### 2.3 Bulding Database of Embeddings

In [ ]:
def build_database():
    with open(TRAIN_JSON) as f:
        data = json.load(f)

    # Map image filenames to their annotations
    img_id_to_name = {img['id']: img['file_name'] for img in data['images']}
    anns_by_img = defaultdict(list)
    for ann in data['annotations']:
        anns_by_img[ann['image_id']].append(ann)

    all_embeddings = []
    all_labels = [] 

    # extract product crops
    for img_id, file_name in tqdm(img_id_to_name.items(), desc="Extracting Training Features"):
        img_path = os.path.join(TRAIN_IMG_DIR, file_name)
        image = cv2.imread(img_path)
        if image is None: 
            continue

        results = yolo_model(image, verbose=False)
        gt_anns = anns_by_img[img_id]

        for r in results:
            for box in r.boxes.xyxy.cpu().numpy():
                x1, y1, x2, y2 = map(int, box)
                crop = image[y1:y2, x1:x2]
                if crop.size == 0: 
                    continue

                best_iou = 0
                best_label = -1
                for ann in gt_anns:
                    gx, gy, gw, gh = ann['bbox']
                    gt_box = [gx, gy, gx+gw, gy+gh]
                    iou = calculate_iou([x1, y1, x2, y2], gt_box)
                    if iou > best_iou:
                        best_iou = iou
                        best_label = ann['category_id']

                if best_iou > 0.6:
                    emb = get_embedding(crop)
                    all_embeddings.append(emb)
                    all_labels.append(int(best_label))

    
    all_embeddings = np.array(all_embeddings).astype('float32')
    d = all_embeddings.shape[1]
    index = faiss.IndexFlatIP(d) 
    index.add(all_embeddings)
    faiss.write_index(index, FAISS_INDEX_PATH)

    with open(LABEL_MAP_PATH, 'w') as f:
        json.dump(all_labels, f)
    
    print(f"Database built with {len(all_labels)} embeddings.")

### 2.4 Metrics Evaluation

In [ ]:
from collections import defaultdict, Counter
import numpy as np

def compute_rpc_metrics(gt_counts, pred_counts):

    categories = set(gt_counts.keys()) | set(pred_counts.keys())

    #  ACD 
    total_gt = sum(gt_counts.values())
    total_pred = sum(pred_counts.values())
    acd = abs(total_gt - total_pred)

    # mCCD
    mccd_list = []
    for c in categories:
        g = gt_counts.get(c, 0)
        p = pred_counts.get(c, 0)

        if g > 0:
            mccd_list.append(abs(p - g) / g)

    mccd = np.mean(mccd_list) if mccd_list else 0

    # mCIoU 
    inter = 0
    union = 0
    for c in categories:
        g = gt_counts.get(c, 0)
        p = pred_counts.get(c, 0)

        inter += min(g, p)
        union += max(g, p)

    mciou = inter / union if union > 0 else 0

    return acd, mccd, mciou

In [ ]:
def evaluate_rpc_metrics():
    print("\n----------------- RPC METRICS (HARD IMAGES) -------------------")

    index = faiss.read_index(FAISS_INDEX_PATH)
    with open(LABEL_MAP_PATH, 'r') as f:
        db_labels = json.load(f)

    with open(TEST_JSON) as f:
        test_data = json.load(f)

    img_map = {img['id']: img for img in test_data['images']}

    ann_by_img = defaultdict(list)
    for ann in test_data['annotations']:
        ann_by_img[ann['image_id']].append(ann)

    # Filter HARD images
    hard_ids = [
        img_id for img_id, img in img_map.items()
        if img.get("level") == "hard" or img.get("difficulty") == "hard"
    ]

    yolo_cAcc = 0
    dino_cAcc = 0

    yolo_acd, dino_acd = [], []
    yolo_mccd, dino_mccd = [], []
    yolo_mciou, dino_mciou = [], []

    total_images = 0

    for img_id in tqdm(hard_ids):

        file_name = img_map[img_id]['file_name']
        img_path = os.path.join(TEST_IMG_DIR, file_name)

        image = cv2.imread(img_path)
        if image is None:
            continue

        gt_anns = ann_by_img[img_id]

        # GT COUNT 
        gt_counts = Counter([ann['category_id'] for ann in gt_anns])


        # YOLO PREDICTION
      
        results = yolo_model(image, verbose=False)

        yolo_labels = []
        for r in results:
            if r.boxes.cls is not None:
                yolo_labels.extend(r.boxes.cls.cpu().numpy().astype(int))

        yolo_counts = Counter(yolo_labels)

        # DINO PREDICTION
        
        dino_labels = []

        for r in results:
            for box in r.boxes.xyxy.cpu().numpy():
                x1, y1, x2, y2 = map(int, box)

                crop = image[y1:y2, x1:x2]
                if crop.size == 0:
                    continue

                emb = get_embedding(crop).reshape(1, -1)
                D, I = index.search(emb, 5)

                labels = [db_labels[i] for i in I[0] if i != -1]

                if labels:
                    pred = Counter(labels).most_common(1)[0][0]
                    dino_labels.append(pred)

        dino_counts = Counter(dino_labels)


        y_acd, y_mccd, y_mciou = compute_rpc_metrics(gt_counts, yolo_counts)
        d_acd, d_mccd, d_mciou = compute_rpc_metrics(gt_counts, dino_counts)

        yolo_acd.append(y_acd)
        yolo_mccd.append(y_mccd)
        yolo_mciou.append(y_mciou)

        dino_acd.append(d_acd)
        dino_mccd.append(d_mccd)
        dino_mciou.append(d_mciou)

        # ===== cAcc =====
        if gt_counts == yolo_counts:
            yolo_cAcc += 1

        if gt_counts == dino_counts:
            dino_cAcc += 1

        total_images += 1


    print("\n----------------- FINAL RESULTS -------------------")

    print("\n--- YOLO ---")
    print(f"cAcc  : {100 * yolo_cAcc / total_images:.2f}%")
    print(f"ACD   : {np.mean(yolo_acd):.2f}")
    print(f"mCCD  : {np.mean(yolo_mccd):.2f}")
    print(f"mCIoU : {100 * np.mean(yolo_mciou):.2f}%")

    print("\n--- DINO ---")
    print(f"cAcc  : {100 * dino_cAcc / total_images:.2f}%")
    print(f"ACD   : {np.mean(dino_acd):.2f}")
    print(f"mCCD  : {np.mean(dino_mccd):.2f}")
    print(f"mCIoU : {100 * np.mean(dino_mciou):.2f}%")

In [ ]:
if __name__ == "__main__":

    build_database()
    evaluate_rpc_metrics()